In [ ]:
import os, re, sqlite3
import pandas as pd
from IPython.display import clear_output
from sentence_transformers import SentenceTransformer

pd.set_option('display.max_colwidth', None)
conn = sqlite3.connect("../data/election.db")
questions = pd.read_sql('SELECT id, axis, text FROM "QuizQuestion" ORDER BY id', conn)
model = SentenceTransformer("intfloat/multilingual-e5-base")
K = 10


def load_segments(party):
    text = open(f'../data/platforms/{party}.txt', encoding='utf-8').read()
    lines = [l.strip() for l in text.split('\n')]
    return [l for l in lines
            if l and not l.startswith('#') and len(l.split()) > 1 and len(l) > 10]


def dedupe(segments, threshold=0.95):
    """מסיר כפילויות מדויקות (אחרי הסרת מספור וסימני פיסוק) וכפילויות כמעט מדויקות."""
    def norm(s):
        s = re.sub(r'^\d+[.)]\s*', '', s)          # "2. חוק גיוס" -> "חוק גיוס"
        return re.sub(r'[^\w\s]', '', s).strip()
    seen, unique = set(), []
    for s in segments:
        if norm(s) not in seen:
            seen.add(norm(s))
            unique.append(s)
    emb = model.encode(["passage: " + s for s in unique], normalize_embeddings=True)
    sims = model.similarity(emb, emb)
    keep = []
    for i in range(len(unique)):
        if not any(sims[i, j] > threshold for j in keep):
            keep.append(i)
    return [unique[i] for i in keep], emb[keep]

In [ ]:
PARTY = 'yashar'
OUT = f'platform_coverage_{PARTY}.csv'

raw = load_segments(PARTY)
segments, passage_emb = dedupe(raw)
query_emb = model.encode(["query: " + t for t in questions['text']], normalize_embeddings=True)
sims = model.similarity(query_emb, passage_emb)
print(f"{PARTY}: {len(raw)} קטעים, {len(segments)} אחרי הסרת כפילויות")

done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
print(f"כבר תויגו: {len(done)} מתוך {len(questions)}")

In [ ]:
done[['question_id', 'stance', 'confidence', 'quote']]

In [ ]:
done = pd.read_csv(OUT)
done = done[~done['question_id'].isin(['q01', 'q03', 'q04'])]
done.to_csv(OUT, index=False)
print(done[['question_id', 'stance']])

In [ ]:
done = pd.read_csv(OUT) if os.path.exists(OUT) else pd.DataFrame()
rows = done.to_dict('records')
answered = {r['question_id'] for r in rows}

for row, q in questions.iterrows():
    if q['id'] in answered:
        continue
    clear_output(wait=True)
    print(f"[{len(rows) + 1}/{len(questions)}]  {q['id']} ({q['axis']}): {q['text']}\n")
    scores, idx = sims[row].topk(K)
    for rank, i in enumerate(idx.tolist(), start=1):
        print(f"{rank:>2}. [{i:>3}] {segments[i]}\n")

    # שואל עד שמתקבלת תשובה תקינה
    while True:
        answer = input("עמדה ומספר קטע (למשל: 1 23 | -1 45 p | 0 12 | Enter = לא מכוסה): ").split()
        if not answer:
            confirm = input(f"{q['id']} לא מכוסה? Enter לאישור, כל דבר אחר כדי לחזור: ")
            if not confirm:
                break
            continue
        if (len(answer) >= 2 and answer[0] in ('1', '-1', '0')
                and answer[1].isdigit() and int(answer[1]) < len(segments)):
            break
        print("לא תקין: קודם כיוון (1, -1 או 0), ואז מספר קטע.")

    if not answer:
        rows.append({'party': PARTY, 'question_id': q['id'], 'axis': q['axis'],
                     'stance': None, 'confidence': 'not_covered', 'passage': None, 'quote': None})
    else:
        stance, passage = int(answer[0]), int(answer[1])
        confidence = 'partial' if 'p' in answer[2:] else ('unclear' if stance == 0 else 'sure')
        rows.append({'party': PARTY, 'question_id': q['id'], 'axis': q['axis'],
                     'stance': stance, 'confidence': confidence,
                     'passage': passage, 'quote': segments[passage]})
    pd.DataFrame(rows).to_csv(OUT, index=False)   # שמירה אחרי כל תשובה

clear_output()
print(f"סיימת! {len(rows)} שאלות נשמרו ב-{OUT}")

In [ ]:
def fix(qid, stance=None, passage=None, partial=False):
    d = pd.read_csv(OUT)
    m = d['question_id'] == qid
    if stance is None:
        d.loc[m, ['stance', 'passage', 'quote']] = None
        d.loc[m, 'confidence'] = 'not_covered'
    else:
        d.loc[m, 'stance'] = stance
        d.loc[m, 'passage'] = passage
        d.loc[m, 'quote'] = segments[passage]
        d.loc[m, 'confidence'] = 'partial' if partial else ('unclear' if stance == 0 else 'sure')
    d.to_csv(OUT, index=False)


# הכיוון נכון, רמת הוודאות לא
fix('q04', 0, 44)                  # שיתופי פעולה אזוריים לא אומרים שנורמליזציה חשובה יותר מהסדר
fix('q05', 1, 35, partial=True)    # "מדיניות ביטחון אגרסיבית" = יד קשה; חסר "גם במחיר בידוד"
fix('q14', -1, 22, partial=True)   # התקצוב מותנה בשירות ולא בלימודי ליבה

# הקטע לא אומר את מה שסומן
fix('q06')                         # ביטול הנחות מס הוא לא הורדת מסים
fix('q08')                         # אכיפה מקומית לא קשורה להפרטה
fix('q17', 0, 61)                  # "תיקונים בהסכמה רחבה" לא מראה כיוון לגבי הוועדה
fix('q21', 0, 32)                  # מסלולי שירות לאומי אינם שוויון בתקציבים; במצע השוויון מותנה

# נשארו כמו שהם בכוונה:
# q01 - קטע 35 ("למנוע מדינה פלסטינית") חזק יותר, אבל זו פסקה ארוכה על הרבה נושאים, אז ציטוט ממנה פחות ממוקד
# q20 - "וועדת חקירה" קצר, אבל הכיוון ברור

d = pd.read_csv(OUT)
print(d['confidence'].value_counts().to_string())
d[d['confidence'] != 'not_covered'][['question_id', 'stance', 'confidence', 'quote']]

In [ ]:
# חלקי: הכיוון נכון אבל הקטע לא תומך בכל ההיגד
fix('q02', 1, 18, partial=True)    # "מרחבי הגבול", לא יהודה ושומרון במפורש
fix('q05', 1, 96, partial=True)    # הכרעה והרתעה, בלי "גם במחיר בידוד"
fix('q06', 1, 38, partial=True)    # שוק חופשי והטבות מס ממוקדות, בלי "על חשבון רווחה"
fix('q12', 1, 23, partial=True)    # שירות לכולם, אבל גם שירות לאומי ולא רק צה"ל
fix('q14', -1, 34, partial=True)   # התקצוב מותנה בתוכן אנטי-ציוני, לא בליבה
fix('q22', 1, 9, partial=True)     # זהות יהודית, אבל לא "גובר על שוויון"

# לא ברור
fix('q21', 0, 10)                  # "זכויות כלל האזרחים", כללי, בלי תקציבים וייצוג

# הכיוון הפוך
fix('q07', -1, 50, partial=True)   # תחרות ופירוק מונופולים במקום התערבות ישירה במחירים

# לא מכוסה
fix('q04')                         # שיתופי פעולה בינלאומיים, לא נורמליזציה מול הסדר
fix('q08')                         # ביזור לרשויות מקומיות אינו הפרטה
fix('q15')                         # הקטע על עסקים קטנים, לא על הרבנות
fix('q17')                         # הקטע לא מזכיר את הוועדה לבחירת שופטים

d = pd.read_csv(OUT)
print(d['confidence'].value_counts().to_string())

In [ ]:
fix('q09', 1, 22)                   # העלאת שכר המינימום ל-70% מהשכר הממוצע
fix('q22', -1, 117, partial=True)   # שוויון במרכז, אבל לא מתייחס לסתירה עם האופי היהודי
fix('q07')                          # הקטע על מכרזים ושחיתות, לא על מחירים
fix('q13')                          # "שוויון אזרחי" אינו נישואין אזרחיים

d = pd.read_csv(OUT)
print(d['confidence'].value_counts().to_string())

In [ ]:
[(i, s) for i, s in enumerate(segments) if 'כולם מתגייסים' in s]

In [ ]:
for word in ['השתמטות', 'פטור', 'גיוס חובה', 'יתגייסו']:
    for i, s in enumerate(segments):
        if word in s:
            print(f"[{i}] {s[:200]}\n")

In [ ]:
fix('q12', 1, 625)

In [ ]:
fix('q14', -1, 350)                  # הכיוון הפוך: חובת ליבה = נגד ההיגד
fix('q04', 1, 638, partial=True)     # נורמליזציה, אבל לא "חשובה יותר מהסדר"
fix('q13', 1, 520, partial=True)     # ברית זוגיות, לא נישואין
fix('q07', 0, 131)                   # כותרת בלי המנגנון
fix('q03')                           # לא מזכיר את תקציב הביטחון
fix('q09')                           # כותרת, לא שכר מינימום
fix('q10')                           # מחירי מזון, לא קצבאות
fix('q18')                           # כספים קואליציוניים, לא היועמ"ש
fix('q21')                           # מרחב ציבורי לנשים, לא אזרחים ערבים
fix('q25')                           # הברחות, לא מבקשי מקלט

d = pd.read_csv(OUT)
print(d['confidence'].value_counts().to_string())

In [ ]:
for word in ['שבת', 'ערבי', 'דרוז', 'חוק הלאום', 'שוויון']:
    print(f"=== {word}")
    for i, s in enumerate(segments):
        if word in s:
            print(f"  [{i}] {s[:180]}")

In [ ]:
# הכיוון הפוך
fix('q14', -1, 69)                   # ליבה כתנאי למימון ציבורי = נגד "ללא דרישת ליבה"
fix('q18', -1, 742, partial=True)    # חיזוק עצמאות היועמ"ש = נגד "לא צריכה לחייב"

# חלקי, בעקביות עם המפלגות הקודמות
fix('q02', 1, 305, partial=True)     # התיישבות בגבולות, לא יהודה ושומרון במפורש
fix('q10', 1, 347, partial=True)     # סיוע בדיור, לא הרחבת קצבאות
fix('q12', 1, 104, partial=True)     # כולם מזומנים, חלקם לשירות אזרחי
fix('q13', 1, 600, partial=True)     # זוגיות אזרחית, לא נישואין
fix('q11', 1, 355, partial=True)     # רשויות מקומיות יעצבו את השבת; לא מזכיר תחבורה במפורש

# לא ברור, עם ציטוט מייצג יותר
fix('q22', 0, 23)                    # "יהודית, דמוקרטית... ברוח מגילת העצמאות" – בלי הכרעה בסתירה

# לא מכוסה
fix('q08')                           # ביזור לרשויות מקומיות אינו הפרטה
fix('q17')                           # רשויות מקומיות, לא הוועדה לבחירת שופטים
fix('q25')                           # עולים, לא מבקשי מקלט
fix('q21')                           # גוף חינוכי עם נציג ערבי אינו שוויון בתקציבים ובייצוג
# fix('q21', 0, 65)                  # ← אם אתה חושב שהקטע כן עוסק בהיגד, השתמש בשורה הזו במקום

d = pd.read_csv(OUT)
print(d['confidence'].value_counts().to_string())
print("\nעם כיוון:", sorted(d[d['confidence'].isin(['sure', 'partial'])]['question_id']))